# attribute demo

In [ ]:
# @title Colab Setup Environment

try:
    import google.colab

    # Remove existing directory to ensure clean clone on re-run
    !rm -rf repository/circuit-tracer

    !mkdir -p repository && cd repository && \
     git clone https://github.com/safety-research/circuit-tracer && \
     curl -LsSf https://astral.sh/uv/install.sh | sh && \
     uv pip install -e circuit-tracer/

    import sys
    from huggingface_hub import notebook_login

    sys.path.append("repository/circuit-tracer")
    sys.path.append("repository/circuit-tracer/demos")
    notebook_login(new_session=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
# ── 1. HuggingFace model for generation ──────────────────────────────────────
from transformers import AutoTokenizer
from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils import create_graph_files
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b")

# ── 2. circuit_tracer model for attribution (replaces HF model entirely) ─────
model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", dtype=torch.bfloat16, backend="transformerlens"
)

In [ ]:
max_n_logits = 10  # How many logits to attribute from, max. We attribute to min(max_n_logits, n_logits_to_reach_desired_log_prob); see below for the latter
desired_logit_prob = 0.95  # Attribution will attribute from the minimum number of logits needed to reach this probability mass (or max_n_logits, whichever is lower)
max_feature_nodes = 8192  # Only attribute from this number of feature nodes, max. Lower is faster, but you will lose more of the graph. None means no limit.
batch_size = 256  # Batch size when attributing
offload = (
    "disk" if IN_COLAB else "cpu"
)  # Offload various parts of the model during attribution to save memory. Can be 'disk', 'cpu', or None (keep on GPU)
verbose = True  # Whether to display a tqdm progress bar and timing report

In [ ]:
# ── 3. Generation loop (same as before, but using ReplacementModel) ───────────
base_prompt = "A rhyming couplet:\nHe saw a carrot and had to grab it,\n"
input_ids = tokenizer(base_prompt, return_tensors="pt")["input_ids"]
STOP_IDS = {108, 235265, tokenizer.eos_token_id}
token_trace = []

for step in range(20):
    with torch.no_grad():
      out = model(input_ids)

    logits_last = out[0, -1, :].float()  # out is already the logits tensor
    probs = F.softmax(logits_last, dim=-1)
    top_probs, top_ids = torch.topk(probs, 10)
    next_id = top_ids[0].item()

    token_trace.append({
        "step": step,
        "token_id": next_id,
        "token_str": tokenizer.decode([next_id]),
        "chosen_prob": top_probs[0].item(),
        "chosen_logit": logits_last[next_id].item(),
        "top10": [{"token": tokenizer.decode([tid.item()]), "prob": p.item()}
                  for tid, p in zip(top_ids, top_probs)],
    })

    if next_id in STOP_IDS:
        break

    input_ids = torch.cat(
        [input_ids, torch.tensor([[next_id]])], dim=1
    )

print("Generated:", "".join(t["token_str"] for t in token_trace if t["token_id"] not in STOP_IDS))

# ── 4. Attribution loop — token_trace flows straight in ──────────────────────
for i, token in enumerate(token_trace):
    if token["token_id"] in STOP_IDS:
        break

    tokens_so_far = "".join(t["token_str"] for t in token_trace[:i])
    prompt_at_step = base_prompt + tokens_so_far

    graph = attribute(
        prompt=prompt_at_step,
        model=model,
        max_n_logits=10,
        desired_logit_prob=0.95,
        batch_size=256,
        offload="cpu",
        verbose=True,
    )

    slug = f"step-{i:02d}-{token['token_str'].strip().replace(' ', '_')}"
    create_graph_files(
        graph_or_path=graph,
        slug=slug,
        output_path="./graph_files",
        node_threshold=0.8,
        edge_threshold=0.98,
    )
    print(f"✓ step {i:02d} → '{token['token_str']}'  saved as '{slug}'")

## chart visualization

In [ ]:
from circuit_tracer.frontend.local_server import serve
from IPython.display import IFrame

port = 8046
server = serve(data_dir="./graph_files/", port=port)

print(f"http://localhost:{port}/index.html")
display(IFrame(src=f"http://localhost:{port}/index.html", width="100%", height="800px"))

In [ ]:
server.stop()